In [2]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
from rich import print as rprint
import os
import time
import sys

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# IMDB + TMDB Dataset
imdb_tmdb_path = hf_hub_download(
    repo_id = "Subhadip007/UERP_Dataset",
    filename = "final_dataset_v1.parquet",
    repo_type = 'dataset',
    token = HF_TOKEN,
)

imdb_tmdb = pd.read_parquet(imdb_tmdb_path)

# Anilist Dataset
anilist_path = hf_hub_download(
    repo_id = "Subhadip007/UERP_Dataset",
    filename = "anilist_final_v1.parquet",
    repo_type = 'dataset',
    token = HF_TOKEN,
)

anilist = pd.read_parquet(anilist_path)

rprint(f"IMDB + TMDB: {imdb_tmdb.shape}")
rprint(f"{imdb_tmdb.columns.tolist()}")
rprint()
rprint(f"Anilist: {anilist.shape}")
rprint(f"{anilist.columns.tolist()}")

final_dataset_v1.parquet:   0%|          | 0.00/10.1M [00:00<?, ?B/s]

anilist_final_v1.parquet:   0%|          | 0.00/2.02M [00:00<?, ?B/s]

IMDB + TMDB: (34765, 15)

['tconst', 'primaryTitle', 'tmdb_id', 'overview', 'poster_path', 'backdrop_path', 'vote_average', 'popularity', 
'genre_ids', 'titleType', 'numVotes', 'startYear', 'genres', 'runtimeMinutes', 'averageRating']

Anilist: (4912, 14)

['anilist_id', 'mal_id', 'title', 'format', 'episodes', 'averageScore', 'popularity', 'favourites', 'seasonYear', 
'genres', 'description', 'image_url', 'isAdult', '_source_page']

In [4]:
display(imdb_tmdb.head(3))
display(anilist.head(3))

,tconst,primaryTitle,tmdb_id,overview,poster_path,backdrop_path,vote_average,popularity,genre_ids,titleType,numVotes,startYear,genres,runtimeMinutes,averageRating
0,tt0111161,The Shawshank Redemption,278,Imprisoned in the 1940s for the double murder ...,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,8.700,62.3273,"[18, 80]",movie,3208129,1994.0,Drama,142,9.3
1,tt0468569,The Dark Knight,155,Batman raises the stakes in his war on crime. ...,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/dqK9Hag1054tghRQSqLSfrkvQnA.jpg,8.533,22.3965,"[28, 80, 53]",movie,3189287,2008.0,"Crime,Thriller",152,9.1
2,tt1375666,Inception,27205,"Cobb, a skilled thief who commits corporate es...",/xlaY2zyzMfkhk0HSC5VUwzoZPU1.jpg,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,8.373,28.9174,"[28, 878, 12]",movie,2835072,2010.0,"Adventure,Sci-Fi,Thriller",148,8.8


,anilist_id,mal_id,title,format,episodes,averageScore,popularity,favourites,seasonYear,genres,description,image_url,isAdult,_source_page
0,16498,16498.0,Attack on Titan,TV,25.0,85.0,1030730,83865,2013.0,"[Action, Drama, Fantasy, Mystery]","Several hundred years ago, humans were nearly ...",https://s4.anilist.co/file/anilistcdn/media/an...,False,1
1,101922,38000.0,Demon Slayer: Kimetsu no Yaiba,TV,26.0,83.0,973809,59248,2019.0,"[Action, Adventure, Drama, Fantasy, Supernatural]","It is the Taisho Period in Japan. Tanjiro, a k...",https://s4.anilist.co/file/anilistcdn/media/an...,False,1
2,113415,40748.0,JUJUTSU KAISEN,TV,24.0,84.0,948353,68386,2020.0,"[Action, Drama, Supernatural]","A boy fights... for ""the right death."" Hardshi...",https://s4.anilist.co/file/anilistcdn/media/an...,False,1


### **Resolving TMDB `genre_ids` before merge**

In [6]:
import requests


def get_tmdb_genre_map(api_key):
    movie_genres = requests.get(
        "https://api.themoviedb.org/3/genre/movie/list",
        params = {
            "api_key": api_key
        }
    ).json()['genres']

    tv_genres = requests.get(
        "https://api.themoviedb.org/3/genre/tv/list",
        params = {
            "api_key": api_key
        }
    ).json()['genres']

    genre_map = {}

    for g in movie_genres + tv_genres:
        genre_map[g['id']] = g['name']
    return genre_map

genre_map = get_tmdb_genre_map(user_secrets.get_secret("TMDB_API_KEY"))
rprint(genre_map)

{
    28: 'Action',
    12: 'Adventure',
    16: 'Animation',
    35: 'Comedy',
    80: 'Crime',
    99: 'Documentary',
    18: 'Drama',
    10751: 'Family',
    14: 'Fantasy',
    36: 'History',
    27: 'Horror',
    10402: 'Music',
    9648: 'Mystery',
    10749: 'Romance',
    878: 'Science Fiction',
    10770: 'TV Movie',
    53: 'Thriller',
    10752: 'War',
    37: 'Western',
    10759: 'Action & Adventure',
    10762: 'Kids',
    10763: 'News',
    10764: 'Reality',
    10765: 'Sci-Fi & Fantasy',
    10766: 'Soap',
    10767: 'Talk',
    10768: 'War & Politics'
}

In [9]:
sample = imdb_tmdb[['primaryTitle', 'genres', 'genre_ids']].head(15)
for _, row in sample.iterrows():
    genre_ids = row['genre_ids']

    if genre_ids is None:
        genre_ids = []
    elif hasattr(genre_ids, "tolist"):
        genre_ids = genre_ids.tolist()
        
    resolved = [genre_map.get(g, f"UNKNOWN({g})") for g in genre_ids]
    
    rprint(f"{row['primaryTitle']}")
    rprint(f"IMDb genres: {row['genres']}")
    rprint(f"TMDB genres: {resolved}")
    rprint()

The Shawshank Redemption

IMDb genres: Drama

TMDB genres: ['Drama', 'Crime']

The Dark Knight

IMDb genres: Crime,Thriller

...

IMDb genres: Adventure,Drama,Fantasy

TMDB genres: ['Adventure', 'Fantasy', 'Action']

Django Unchained

IMDb genres: Drama,Western

TMDB genres: ['Drama', 'Western']

In [10]:
imdb_genre_unique = set()

for g in imdb_tmdb['genres'].dropna():
    imdb_genre_unique.update(g.split(","))

rprint(f"IMDB unique genres: {sorted(imdb_genre_unique)}")
rprint()

rprint(f"TMDB unique genres: {sorted(genre_map.values())}")
rprint()

anilist_genre_unique = set()

for g in anilist['genres'].dropna():
    anilist_genre_unique.update(g)

rprint(f"Anilist unique genres: {sorted(anilist_genre_unique)}")

IMDB unique genres: ['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 
'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News',
'Reality-TV', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']

TMDB unique genres: ['Action', 'Action & Adventure', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 
'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Kids', 'Music', 'Mystery', 'News', 'Reality', 'Romance', 
'Sci-Fi & Fantasy', 'Science Fiction', 'Soap', 'TV Movie', 'Talk', 'Thriller', 'War', 'War & Politics', 'Western']

Anilist unique genres: ['Action', 'Adventure', 'Comedy', 'Drama', 'Ecchi', 'Fantasy', 'Horror', 'Mahou Shoujo', 
'Mecha', 'Music', 'Mystery', 'Psychological', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 
'Thriller']

In [12]:
[col for col in imdb_tmdb.columns if "adult" in col.lower()]

[]

In [13]:
[col for col in imdb_tmdb.columns if "isadult" in col.lower()]

[]

In [17]:
imdb_tmdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34765 entries, 0 to 34764
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tconst          34765 non-null  object 
 1   primaryTitle    34765 non-null  object 
 2   tmdb_id         34765 non-null  int64  
 3   overview        34765 non-null  object 
 4   poster_path     34673 non-null  object 
 5   backdrop_path   34118 non-null  object 
 6   vote_average    34759 non-null  float64
 7   popularity      34759 non-null  float64
 8   genre_ids       34759 non-null  object 
 9   titleType       34765 non-null  object 
 10  numVotes        34765 non-null  int64  
 11  startYear       34765 non-null  float64
 12  genres          34751 non-null  object 
 13  runtimeMinutes  33705 non-null  object 
 14  averageRating   34765 non-null  float64
dtypes: float64(4), int64(2), object(9)
memory usage: 4.0+ MB


In [19]:
try:
    rprint(imdb_tmdb['isAdult'].value_counts())
    rprint()

    adult_titles = imdb_tmdb[imdb_tmdb['isAdult'].astype(str).isin(['1', '1.0', 'True'])]
    rprint(f"Adult-flagged titles in our catalog: {len(adult_titles)}")
    rprint(f"{adult_titles[['primaryTitle', 'genres']].head(10)}")
except Exception as e:
    rprint(f"No Column name: {e}")

No Column name: 'isAdult'

In [20]:
import pandas as pd
import urllib.request
import os

os.makedirs('/kaggle/working/data', exist_ok = True)

basics_path = "/kaggle/working/data/title.basics.tsv.gz"

if not os.path.exists(basics_path):
    rprint(f"Downloading title.basics.tsv.gz")
    urllib.request.urlretrieve('https://datasets.imdbws.com/title.basics.tsv.gz', basics_path)
    rprint('Done.')
else:
    rprint("Already exists, skipping download.")

isadult_lookup = pd.read_csv(
    basics_path,
    sep = '\t',
    na_values = '\\N',
    usecols = ['tconst', 'isAdult'],
    low_memory = False,
)

rprint(isadult_lookup.shape)
rprint(isadult_lookup['isAdult'].value_counts())

Downloading title.basics.tsv.gz

Done.

(12660112, 2)

isAdult
0       12247562
1         412519
2023           7
1979           4
2019           3
2024           3
2025           3
2018           2
2022           2
2020           2
1981           1
2017           1
2014           1
2013           1
2005           1
Name: count, dtype: int64

In [21]:
imdb_tmdb = imdb_tmdb.merge(isadult_lookup, on = 'tconst', how = 'left')
rprint(imdb_tmdb['isAdult'].value_counts())

adult_titles = imdb_tmdb[imdb_tmdb['isAdult'] == 1]
rprint("Adult-flagged titles in catalog:", len(adult_titles))
rprint(adult_titles[['primaryTitle', 'genres']].head(10))

isAdult
0    34763
1        2
Name: count, dtype: int64

Adult-flagged titles in catalog: 2

primaryTitle               genres
15113                  Deep Throat         Adult,Comedy
32852  Gayniggers from Outer Space  Comedy,Sci-Fi,Short

In [22]:
imdb_tmdb_clean = imdb_tmdb[imdb_tmdb['isAdult'] != 1].copy()
rprint("Before:", imdb_tmdb.shape[0], "| After:", imdb_tmdb_clean.shape[0])

imdb_tmdb = imdb_tmdb_clean

Before: 34765 | After: 34763

In [23]:
GENRE_MAP = {
    # renames
    'Sci-Fi': 'Science Fiction',
    'Sport': 'Sports',
    'Reality-TV': 'Reality',
    'Talk-Show': 'Talk Show',
    'Talk': 'Talk Show',
    # merges
    'Kids': 'Family',
    'Soap': 'Drama',
    # drops (content-type tags, genre nahi)
    'Short': None,
    'TV Movie': None,
    'Adult': None,
}

SPLIT_MAP = {
    'Action & Adventure': ['Action', 'Adventure'],
    'Sci-Fi & Fantasy': ['Science Fiction', 'Fantasy'],
    'War & Politics': ['War'],
}

In [24]:
def normalize_genre_list(raw_genres):
    result = set()
    for g in raw_genres:
        if g in SPLIT_MAP:
            result.update(SPLIT_MAP[g])
        elif g in GENRE_MAP:
            mapped = GENRE_MAP[g]
            if mapped is not None:
                result.add(mapped)
        else:
            result.add(g)
    return sorted(result)

In [26]:
imdb_tmdb['genres_canonical'] = imdb_tmdb['genres'].fillna("").apply(
    lambda x: normalize_genre_list(x.split(",")) if x else []
)

anilist['genres_canonical'] = anilist['genres'].apply(normalize_genre_list)

all_canonical = set()

for g in imdb_tmdb['genres_canonical']:
    all_canonical.update(g)

for g in anilist['genres_canonical']:
    all_canonical.update(g)

rprint(f"Final canonical genre list: {sorted(all_canonical)}")
rprint(f"Total canonical genres: {len(all_canonical)}")

Final canonical genre list: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 
'Drama', 'Ecchi', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Mahou Shoujo', 'Mecha', 
'Music', 'Musical', 'Mystery', 'News', 'Psychological', 'Reality', 'Romance', 'Science Fiction', 'Slice of Life', 
'Sports', 'Supernatural', 'Talk Show', 'Thriller', 'War', 'Western']

Total canonical genres: 32

In [28]:
TMDB_IMAGE_BASE = "https://image.tmdb.org/t/p/w500"

CONTENT_TYPE_MAP_IMDB = {
    'movie': 'movie', 
    'tvSeries': 'tv_series', 
    'tvMiniSeries': 'tv_miniseries',
    'tvMovie': 'tv_movie', 
    'tvSpecial': 'tv_special', 
    'short': 'short', 
    'tvShort': 'short',
}

CONTENT_TYPE_MAP_ANILIST = {
    'MOVIE': 'movie', 
    'TV': 'tv_series', 
    'OVA': 'tv_special',
    'ONA': 'tv_special', 
    'SPECIAL': 'tv_special', 
    'TV_SHORT': 'short',
}

imdb_unified = pd.DataFrame(
    {
        'content_id': 'imdb_' + imdb_tmdb['tconst'],
        'is_anime': False,
        'title': imdb_tmdb['primaryTitle'],
        'content_type': imdb_tmdb['titleType'].map(CONTENT_TYPE_MAP_IMDB),
        'year': imdb_tmdb['startYear'],
        'genres': imdb_tmdb['genres_canonical'],
        'overview': imdb_tmdb['overview'],
        'rating_normalized': imdb_tmdb['averageRating'],
        'popularity_signal': imdb_tmdb['numVotes'],
        'poster_url': TMDB_IMAGE_BASE + imdb_tmdb['poster_path'].fillna(''),
        'runtime_minutes': imdb_tmdb['runtimeMinutes'],
        'episodes': None,
    }
)

imdb_unified.loc[imdb_tmdb['poster_path'].isna(), 'poster_url'] = None

anilist_unified = pd.DataFrame(
    {
        'content_id': 'anilist_' + anilist['anilist_id'].astype(str),
        'is_anime': True,
        'title': anilist['title'],
        'content_type': anilist['format'].map(CONTENT_TYPE_MAP_ANILIST),
        'year': anilist['seasonYear'],
        'genres': anilist['genres_canonical'],
        'overview': anilist['description'],
        'rating_normalized': anilist['averageScore'] / 10.0,
        'popularity_signal': anilist['popularity'],
        'poster_url': anilist['image_url'],
        'runtime_minutes': None,
        'episodes': anilist['episodes'],
    }
)

imdb_unified['episodes'] = pd.array([None] * len(imdb_unified), dtype = 'Int64')
anilist_unified['runtime_minutes'] = pd.array([None] * len(anilist_unified), dtype = 'Int64')

unified_catalog = pd.concat([imdb_unified, anilist_unified], ignore_index = True)


rprint("Total rows:", unified_catalog.shape)
rprint()
rprint("content_type distribution:")
rprint(unified_catalog['content_type'].value_counts())
rprint()
rprint("Nulls per column:")
rprint(unified_catalog.isnull().sum())
rprint()
rprint("Sample rows:")
rprint(unified_catalog.sample(5)[['content_id', 'title', 'is_anime', 'content_type', 'genres', 'rating_normalized']])

/tmp/ipykernel_58/3750303108.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unified_catalog = pd.concat([imdb_unified, anilist_unified], ignore_index = True)


Total rows:
(39675, 12)

content_type distribution:

content_type
movie            20660
tv_series        10741
tv_special        2183
short             2156
tv_movie          1968
tv_miniseries     1956
Name: count, dtype: int64

Nulls per column:

content_id               0
is_anime                 0
title                    0
content_type            11
year                   171
genres                   0
overview                 4
rating_normalized      104
popularity_signal        0
poster_url              92
runtime_minutes       5972
episodes             34877
dtype: int64

Sample rows:

content_id               title  is_anime content_type  \
2974    imdb_tt1261945  Sex and the City 2     False        movie   
26774  imdb_tt23772966      To Cook a Bear     False    tv_series   
15107  imdb_tt10738536           Delicious     False        movie   
7347    imdb_tt8836988          Avengement     False        movie   
24608  imdb_tt17372734             Saint X     False    tv_series   

                           genres  rating_normalized  
2974     [Comedy, Drama, Romance]                4.6  
26774     [Crime, Drama, Mystery]                7.2  
15107    [Comedy, Drama, History]                7.0  
7347    [Action, Crime, Thriller]                6.6  
24608  [Drama, Mystery, Thriller]                6.4

In [29]:
null_ct = unified_catalog[unified_catalog['content_type'].isnull()]
rprint(null_ct[['content_id', 'title', 'is_anime']])
rprint()
rprint("is_anime breakdown of nulls:", null_ct['is_anime'].value_counts())

content_id                                              title  \
37219  anilist_204429                          Chainsaw Man: Shikaku-hen   
37367  anilist_171630                                      THE ONE PIECE   
37638  anilist_209895           Jujutsu Kaisen: Shimetsu Kaiyuu - Kouhen   
37789  anilist_161410                Lycoris Recoil (Shinsaku Animation)   
37850  anilist_154765                                            Genshin   
38725  anilist_204362                         Kaijuu 8-gou: Kanketsu-hen   
39014  anilist_195448  Monogatari Series: Off & Monster Season Shin E...   
39089  anilist_206812                    Dragon Ball Super: Ginga Patrol   
39249  anilist_196222  Ore wa Subete wo [Parry] Suru: Gyaku Kanchigai...   
39438  anilist_208192                       Evangelion (Shinsaku Series)   
39657  anilist_204293                        Akatsuki no Yona (Zoku-hen)   

       is_anime  
37219      True  
37367      True  
37638      True  
37789      True  
37850      True  
38725      True  
39014      True  
39089      True  
39249      True  
39438      True  
39657      True

is_anime breakdown of nulls: is_anime
True    11
Name: count, dtype: int64

In [30]:
null_ids = null_ct['content_id'].str.replace('anilist_', '').astype(int)
check = anilist[anilist['anilist_id'].isin(null_ids)][['anilist_id', 'title', 'format']]
rprint(check)

anilist_id                                              title format
2456      204429                          Chainsaw Man: Shikaku-hen   None
2604      171630                                      THE ONE PIECE   None
2875      209895           Jujutsu Kaisen: Shimetsu Kaiyuu - Kouhen   None
3026      161410                Lycoris Recoil (Shinsaku Animation)   None
3087      154765                                            Genshin   None
3962      204362                         Kaijuu 8-gou: Kanketsu-hen   None
4251      195448  Monogatari Series: Off & Monster Season Shin E...   None
4326      206812                    Dragon Ball Super: Ginga Patrol   None
4486      196222  Ore wa Subete wo [Parry] Suru: Gyaku Kanchigai...   None
4675      208192                       Evangelion (Shinsaku Series)   None
4894      204293                        Akatsuki no Yona (Zoku-hen)   None

In [31]:
unified_catalog = unified_catalog[unified_catalog['content_type'].notnull()].copy()
rprint("Final catalog size:", unified_catalog.shape)

Final catalog size:
(39664, 12)

In [34]:
display(unified_catalog.head())
rprint(unified_catalog.shape)

,content_id,is_anime,title,content_type,year,genres,overview,rating_normalized,popularity_signal,poster_url,runtime_minutes,episodes
0,imdb_tt0111161,False,The Shawshank Redemption,movie,1994.0,[Drama],Imprisoned in the 1940s for the double murder ...,9.3,3208129,https://image.tmdb.org/t/p/w500/9cqNxx0GxF0bfl...,142,NaN
1,imdb_tt0468569,False,The Dark Knight,movie,2008.0,"[Crime, Thriller]",Batman raises the stakes in his war on crime. ...,9.1,3189287,https://image.tmdb.org/t/p/w500/qJ2tW6WMUDux91...,152,NaN
2,imdb_tt1375666,False,Inception,movie,2010.0,"[Adventure, Science Fiction, Thriller]","Cobb, a skilled thief who commits corporate es...",8.8,2835072,https://image.tmdb.org/t/p/w500/xlaY2zyzMfkhk0...,148,NaN
3,imdb_tt0137523,False,Fight Club,movie,1999.0,"[Crime, Drama, Thriller]",A ticking-time-bomb insomniac and a slippery s...,8.8,2630140,https://image.tmdb.org/t/p/w500/jSziioSwPVrOy9...,139,NaN
4,imdb_tt0816692,False,Interstellar,movie,2014.0,"[Adventure, Drama, Science Fiction]",The adventures of a group of explorers who mak...,8.7,2559295,https://image.tmdb.org/t/p/w500/yQvGrMoipbRodd...,169,NaN


(39664, 12)

In [36]:
from huggingface_hub import login, HfApi

login(token = HF_TOKEN)
api = HfApi()

os.makedirs('/kaggle/working/processed', exist_ok = True)
unified_catalog.to_parquet('/kaggle/working/processed/unified_catalog_v1.parquet', index = False)

api.upload_file(
    path_or_fileobj = '/kaggle/working/processed/unified_catalog_v1.parquet',
    path_in_repo = 'unified_catalog_v1.parquet',
    repo_id = 'Subhadip007/UERP_Dataset',
    repo_type = 'dataset',
)

rprint("Unified catalog pushed to HF Hub!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unified catalog pushed to HF Hub!